# Week 8 Lab — Predator-Prey Dynamics and Differential Equations
## SCIE1500 Science and Quantitative Reasoning — Semester 2, 2026

**Lab format:** Work in your groups. Run all code cells in order, fill in the written prompts.

---
**This week:** We model two interacting populations using a system of ordinary differential equations (ODEs) — the Lotka-Volterra predator-prey model.

---
## 📋 Scientific Scenario

Feral cats threaten numbats in the Western Australian wheatbelt. The population dynamics can be modelled as:

$$\frac{dN}{dt} = 0.4N - 0.002NC$$

$$\frac{dC}{dt} = -0.3C + 0.001NC$$

where $N$ is the numbat population and $C$ is the feral cat population.

- The $+0.4N$ term: numbats grow at 40%/yr in the absence of cats.
- The $-0.002NC$ term: cats reduce numbat growth (predation rate).
- The $-0.3C$ term: cats decline at 30%/yr without numbats.
- The $+0.001NC$ term: cat population grows from numbat predation.

---
## 🎯 Group Task & Learning Objectives

| Part | Topic | Time |
|------|-------|------|
| A | Find equilibria and derive the nullclines algebraically | ~20 min |
| B | Plot the phase plane and interpret the vector field | ~20 min |
| C | Simulate and plot population trajectories over time | ~20 min |

By the end you should be able to: find equilibria of ODE systems algebraically; plot phase planes and nullclines; and simulate predator-prey dynamics numerically.

In [ ]:
# Run this cell first — loads libraries for today's lab
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Lotka-Volterra parameters
r_N = 0.4    # numbat growth rate
a   = 0.002  # predation rate (cats kill numbats)
d   = 0.3    # cat death rate
b   = 0.001  # cat growth from predation

# Equilibrium: set dN/dt = 0 AND dC/dt = 0
# dN/dt = 0: N(0.4 - 0.002C) = 0 → C* = 0.4/0.002 = 200
# dC/dt = 0: C(-0.3 + 0.001N) = 0 → N* = 0.3/0.001 = 300
C_star = r_N / a
N_star = d   / b
print(f"Coexistence equilibrium: N* = {N_star:.0f} numbats, C* = {C_star:.0f} cats")
print(f"Trivial equilibrium:     N = 0, C = 0")

---
## Part A: Equilibrium and Nullclines (~20 min)

The **nullclines** are curves where one population's rate of change is zero:

- **Numbat nullcline** ($dN/dt = 0$): $C = 200$ (horizontal line)
- **Cat nullcline** ($dC/dt = 0$): $N = 300$ (vertical line)

✏️ **A.1** What happens to the numbat population when $C > 200$? When $C < 200$?

✏️ **A.2** What happens to the cat population when $N > 300$? When $N < 300$?

```
ANSWERS:
...
```

In [ ]:
# A.1 — Check rates at a point to understand direction
# Start: N=400 numbats, C=100 cats
N, C = 400, 100
dN = r_N*N - a*N*C
dC = -d*C  + b*N*C
print(f"At N={N}, C={C}:")
print(f"  dN/dt = {dN:.1f}  ({'increasing' if dN > 0 else 'decreasing'})")
print(f"  dC/dt = {dC:.1f}  ({'increasing' if dC > 0 else 'decreasing'})")

# Try another point
N, C = 200, 300
dN = r_N*N - a*N*C
dC = -d*C  + b*N*C
print(f"\nAt N={N}, C={C}:")
print(f"  dN/dt = {dN:.1f}  ({'increasing' if dN > 0 else 'decreasing'})")
print(f"  dC/dt = {dC:.1f}  ({'increasing' if dC > 0 else 'decreasing'})")

---
## Part B: Phase Plane (~20 min)

In [ ]:
# B.1 — Phase plane with quiver (vector field)
N_grid = np.linspace(0, 600, 20)
C_grid = np.linspace(0, 400, 20)
NN, CC = np.meshgrid(N_grid, C_grid)

dN_dt = r_N*NN - a*NN*CC
dC_dt = -d*CC  + b*NN*CC

# Normalise arrows for display
magnitude = np.sqrt(dN_dt**2 + dC_dt**2) + 1e-6
dN_norm = dN_dt / magnitude
dC_norm = dC_dt / magnitude

fig, ax = plt.subplots(figsize=(8, 6))
ax.quiver(NN, CC, dN_norm, dC_norm, magnitude, cmap="RdYlGn", alpha=0.7)

# Nullclines
ax.axvline(N_star, color="steelblue", lw=2, ls="--", label=f"Cat nullcline: N={N_star:.0f}")
ax.axhline(C_star, color="firebrick", lw=2, ls="--", label=f"Numbat nullcline: C={C_star:.0f}")
ax.plot(N_star, C_star, "ko", ms=14, zorder=5, label=f"Coexistence eq. ({N_star:.0f},{C_star:.0f})")

ax.set_xlabel("Numbat Population N")
ax.set_ylabel("Feral Cat Population C")
ax.set_title("Feral Cat–Numbat Phase Plane")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part C: Population Trajectories (~20 min)

In [ ]:
# C.1 — Simulate using Euler's method
dt   = 0.01   # time step (years)
T    = 30     # total time (years)
steps = int(T / dt)

N0, C0 = 400, 80   # initial populations

N_traj = np.zeros(steps + 1)
C_traj = np.zeros(steps + 1)
t_traj = np.zeros(steps + 1)

N_traj[0], C_traj[0] = N0, C0

for i in range(steps):
    dN = (r_N * N_traj[i] - a * N_traj[i] * C_traj[i]) * dt
    dC = (-d  * C_traj[i] + b * N_traj[i] * C_traj[i]) * dt
    N_traj[i+1] = N_traj[i] + dN
    C_traj[i+1] = C_traj[i] + dC
    t_traj[i+1] = t_traj[i] + dt

print(f"Initial:  N={N0}, C={C0}")
print(f"Year 10:  N={N_traj[int(10/dt)]:.0f}, C={C_traj[int(10/dt)]:.0f}")
print(f"Year 20:  N={N_traj[int(20/dt)]:.0f}, C={C_traj[int(20/dt)]:.0f}")

In [ ]:
# C.2 — Plot time series
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(t_traj, N_traj, "steelblue", lw=2, label="Numbats N(t)")
ax1.axhline(N_star, color="steelblue", ls="--", alpha=0.4, label=f"Equilibrium N*={N_star:.0f}")
ax1.set_ylabel("Numbat Population")
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

ax2.plot(t_traj, C_traj, "firebrick", lw=2, label="Feral Cats C(t)")
ax2.axhline(C_star, color="firebrick", ls="--", alpha=0.4, label=f"Equilibrium C*={C_star:.0f}")
ax2.set_xlabel("Years"); ax2.set_ylabel("Cat Population")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.suptitle("Numbat–Feral Cat Population Dynamics (N₀=400, C₀=80)")
plt.tight_layout()
plt.show()

In [ ]:
# C.3 — Management scenario: cat culling + numbat translocation
print("Management scenario: remove 50 cats/year, add 20 numbats/year")
N_mgmt = np.zeros(steps + 1)
C_mgmt = np.zeros(steps + 1)
N_mgmt[0], C_mgmt[0] = 200, 250   # low numbats, high cats (stressed state)

for i in range(steps):
    dN = (r_N * N_mgmt[i] - a * N_mgmt[i] * C_mgmt[i] + 20) * dt
    dC = (-d  * C_mgmt[i] + b * N_mgmt[i] * C_mgmt[i] - 50) * dt
    N_mgmt[i+1] = max(0, N_mgmt[i] + dN)
    C_mgmt[i+1] = max(0, C_mgmt[i] + dC)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_traj, N_mgmt, "steelblue", lw=2, label="Numbats (managed)")
ax.plot(t_traj, C_mgmt, "firebrick", lw=2, label="Feral Cats (managed)")
ax.axhline(N_star, color="steelblue", ls=":", alpha=0.5)
ax.axhline(C_star, color="firebrick", ls=":", alpha=0.5)
ax.set_xlabel("Years"); ax.set_ylabel("Population")
ax.set_title("Managed Scenario: Culling 50 Cats/yr + 20 Numbat Translocations/yr")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

✏️ **Group Discussion:**

1. Do the populations oscillate or converge to equilibrium? How does your phase plane explain this?
2. Under the management scenario, does the numbat population recover? What is the long-run outcome?
3. What are the limitations of the Lotka-Volterra model for real conservation planning?

```
DISCUSSION:
...
```